# Q25: Masked Attention
**Topic:** LLM | **Difficulty:** Medium | **Time:** 25 min
**Sheet:** GrindLLM50

---

## Question
How and why is 'masked' attention implemented?

## Detailed Answer

### Types of Masking

**1. Causal (Autoregressive) Mask**
Prevents attending to **future tokens**.
```
Mask = upper_triangular(ones) → -inf
[1  -∞  -∞  -∞]    Token 1 sees: [1]
[1   1  -∞  -∞]    Token 2 sees: [1, 2]
[1   1   1  -∞]    Token 3 sees: [1, 2, 3]
[1   1   1   1]    Token 4 sees: [1, 2, 3, 4]
```
- Essential for language generation (can't look ahead)
- Used in GPT, LLaMA, all decoder-only models

**2. Padding Mask**
Prevents attending to **padding tokens** in batched sequences.
```
Sequence: [The, cat, sat, <PAD>, <PAD>]
Mask:     [1,   1,   1,   -∞,    -∞]
```

**3. MLM Mask (BERT)**
Masks 15% of tokens for prediction:
- 80% replaced with [MASK]
- 10% replaced with random token
- 10% kept as-is

### Implementation
```python
# Causal mask
mask = torch.triu(torch.ones(n, n), diagonal=1).bool()  # upper triangle
scores = scores.masked_fill(mask, float('-inf'))  # -inf before softmax
attn_weights = torch.softmax(scores, dim=-1)  # -inf → 0 probability
```

### Why -inf?
$\text{softmax}(-\infty) = 0$ — masked positions contribute exactly zero weight.
This is mathematically equivalent to removing those tokens from the attention computation.

### Why Masking Matters
- **Causal**: Without it, the model would "cheat" by seeing the answer during training
- **Padding**: Without it, PAD tokens would distort attention weights
- Masking is applied **before softmax**: $\text{softmax}(QK^T/\sqrt{d_k} + M)$ where M contains 0 or -∞

## Key Takeaways
- **Causal mask** = upper triangle of -∞ → prevents seeing future tokens
- **Padding mask** = -∞ for PAD positions → prevents attending to padding
- Applied **before softmax**: $e^{-\infty} = 0$ gives zero attention weight
- Without causal masking, autoregressive training would be impossible